First, let's install the necessary Python libraries: `transformers`, `torch`, `pandas`, and `statsmodels`.

In [1]:
pip install transformers torch pandas statsmodels

In [2]:
!pip install bitsandbytes

In [3]:
!pip install -q transformers accelerate bitsandbytes

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Install/upgrade bitsandbytes and accelerate. A runtime restart is often required after this step.
!pip install -q --upgrade bitsandbytes accelerate

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CHECKPOINT = "allenai/OLMo-2-1124-7B-DPO"
FORMAT = "chat"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=quantization_config,
    device_map="auto",
)

model.eval()

print("Model loaded:", CHECKPOINT)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 8.9 MB/s eta 0:00:00


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/31.0k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

Model loaded: allenai/OLMo-2-1124-7B-DPO


In [5]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

In [6]:
with open(
    "/content/drive/MyDrive/Colab Notebooks/"
    "pragmatics_project/redundancy_probes.json"
) as f:
    rows = json.load(f)

item1_rows = [
    r for r in rows
    if r["item_id"] == 1
]

print(f"Loaded {len(item1_rows)} conditions")

Loaded 4 conditions


In [7]:
"""
Log-probability forced-choice scoring, to replace free-text generation
for probes showing bias (e.g. redundancy_id always answering the same
option regardless of content).

Instead of generating and parsing text, this computes the model's
log-probability of each candidate continuation directly and picks
whichever is higher -- removes surface generation artifacts (option-
order bias, hedging, stray tokens) entirely.
"""

import torch


def score_continuation(model, tokenizer, prompt, continuation):
    """Return the summed log-probability the model assigns to
    `continuation` appearing right after `prompt`."""
    full_text = prompt + continuation
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    full_ids = tokenizer(full_text, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        outputs = model(full_ids)
        logits = outputs.logits[0]  # (seq_len, vocab_size)

    # log-prob of each continuation token, given everything before it
    log_probs = torch.log_softmax(logits, dim=-1)
    continuation_start = prompt_ids.shape[1]
    total_log_prob = 0.0
    for i in range(continuation_start, full_ids.shape[1]):
        token_id = full_ids[0, i]
        total_log_prob += log_probs[i - 1, token_id].item()

    return total_log_prob


def forced_choice(model, tokenizer, prompt, option_a, option_b):
    """Return whichever of option_a/option_b has higher log-probability,
    plus both raw scores (log the scores -- useful for spotting
    borderline/uncertain cases later during error analysis)."""
    score_a = score_continuation(model, tokenizer, prompt, option_a)
    score_b = score_continuation(model, tokenizer, prompt, option_b)
    winner = option_a if score_a > score_b else option_b
    return winner, {"option_a": option_a, "score_a": score_a,
                     "option_b": option_b, "score_b": score_b}


def mcq_choice(model, tokenizer, prompt, letter_map):
    """
    Score a lettered MCQ probe (single-token A/B comparison -- avoids
    surface form competition from comparing full phrases). letter_map
    is the {"A": "meaning", "B": "meaning"} dict generate_probes.py
    stores alongside each mcq prompt. Returns the decoded meaning
    (not just the letter) plus raw scores for error analysis.
    """
    winner_letter_str, scores = forced_choice(model, tokenizer, prompt, " A", " B")
    winner_letter = winner_letter_str.strip()  # "A" or "B"
    winner_meaning = letter_map[winner_letter]
    return winner_meaning, winner_letter, scores


# --- Example usage for redundancy_id specifically ---------------------
# prompt should end right where the answer begins, e.g. "...Answer:"
# option_a / option_b should include the leading space, e.g. " New information"

# winner, scores = forced_choice(
#     model, tokenizer,
#     prompt="... Answer with exactly one phrase: New information or Already expected.\nAnswer:",
#     option_a=" New information",
#     option_b=" Already expected",
# )

In [ ]:
def generate(prompt, max_new_tokens=15):
    if FORMAT == "chat":
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        input_ids = tokenizer(formatted_prompt, return_tensors="pt").input_ids.to(model.device)
    else:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # greedy -- deterministic for piloting
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def format_for_scoring(prompt):
    """Apply the same chat-template formatting generate() uses, so the
    log-prob comparison is scored on identical input to what generate()
    saw. Returns the formatted string ready to append an answer to."""
    if FORMAT == "chat":
        messages = [{"role": "user", "content": prompt}]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        return prompt


for row in item1_rows:
    print("\n" + "=" * 70)
    print(f"CONDITION: {row['condition']}")
    for step in ["script_knowledge", "redundancy_id", "inference_derivation"]:
        mcq_prompt = row["probes"][step]["mcq"]
        letter_map = row["probes"][step]["mcq_letter_map"]
        print(f"\n--- {step} ---")
        print(f"MCQ PROMPT (last 250 chars): ...{mcq_prompt[-250:]}")
        print(f"letter_map: {letter_map}")

        # all three steps now use MCQ log-prob scoring (single-token
        # letter comparison), avoiding surface form competition
        formatted_prompt = format_for_scoring(mcq_prompt)
        winner_meaning, winner_letter, scores = mcq_choice(
            model, tokenizer, formatted_prompt, letter_map
        )
        print(f"WINNER: {winner_letter} = {winner_meaning!r}")
        print(f"  scores: {scores}")


CONDITION: critical

--- script_knowledge ---
MCQ PROMPT (last 250 chars): ...John often goes to the grocery store around the corner from his apartment.
Question: Based on this, is it likely or unlikely that he pays the cashier when he goes to the grocery store?
(A) Unlikely
(B) Likely
Answer with the letter only.
Answer:
letter_map: {'A': 'unlikely', 'B': 'likely'}
WINNER: B = 'likely'
  scores: {'option_a': ' A', 'score_a': -19.875, 'option_b': ' B', 'score_b': -12.4375}

--- redundancy_id ---
MCQ PROMPT (last 250 chars): ...ust came back from the grocery store. He paid the cashier."
Question: Is the detail "He paid the cashier." new information, or something already expected given the context?
(A) Already expected
(B) New information
Answer with the letter only.
Answer:
letter_map: {'A': 'expected', 'B': 'new'}
WINNER: A = 'expected'
  scores: {'option_a': ' A', 'score_a': -14.875, 'option_b': ' B', 'score_b': -16.0}

--- inference_derivation ---
MCQ PROMPT (last 250 chars): ...er:

In [ ]:
for row in item1_rows:
    print("\n" + "=" * 70)
    print(f"CONDITION: {row['condition']}")
    for step in ["script_knowledge", "redundancy_id", "inference_derivation"]:
        prompt = row["probes"][step][FORMAT]
        print(f"\n--- {step} ---")
        print(f"PROMPT (last 200 chars): ...{prompt[-200:]}")

        if step == "redundancy_id":
            # log-prob scoring instead of generation, to bypass the
            # position/format bias seen with free-text generation
            formatted_prompt = format_for_scoring(prompt)
            winner, scores = forced_choice(
                model, tokenizer, formatted_prompt, *REDUNDANCY_OPTIONS
            )
            print(f"LOG-PROB WINNER: {winner!r}")
            print(f"  scores: {scores}")
        else:
            response = generate(prompt)
            print(f"RAW OUTPUT: {response!r}")

In [8]:
print(mcq_choice)

<function mcq_choice at 0x7c7f2106ac00>


In [10]:
import json

for stage in ["pretrained", "sft", "dpo"]:
    path = f"/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results.json/{stage}.jsonl"
    total, failed = 0, 0
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            total += 1
            if r["error"] is not None:
                failed += 1
    print(f"{stage}: {failed}/{total} rows failed")

pretrained: 288/288 rows failed
sft: 288/288 rows failed
dpo: 288/288 rows failed


In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json


CHECKPOINTS = {
    "pretrained": {"name": "allenai/OLMo-2-1124-7B", "format": "completion"},
    "sft":        {"name": "allenai/OLMo-2-1124-7B-SFT", "format": "chat"},
    "dpo":        {"name": "allenai/OLMo-2-1124-7B-DPO", "format": "chat"},
}

RUN_ALL_CHECKPOINTS = False
CHECKPOINT_TO_RUN = "pretrained"  # used only if RUN_ALL_CHECKPOINTS is False

PROBES_PATH = "/content/drive/MyDrive/Colab Notebooks/pragmatics_project/redundancy_probes.json"
RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results.json"
os.makedirs(RESULTS_DIR, exist_ok=True)

STEPS = ["script_knowledge", "redundancy_id", "inference_derivation"]
RELIABLE_STEPS = {"script_knowledge", "inference_derivation"}  # redundancy_id parked


def load_model(checkpoint_name):
    print(f"Loading {checkpoint_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

    # Use BitsAndBytesConfig for 8-bit loading
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        checkpoint_name,
        quantization_config=quantization_config,
        device_map="auto",
    )
    return model, tokenizer


def format_for_scoring(prompt, fmt, tokenizer):
    """Apply chat-template formatting for chat-format checkpoints;
    pass through unchanged for completion-format (base) checkpoints."""
    if fmt == "chat":
        messages = [{"role": "user", "content": prompt}]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        # base/completion checkpoints: MCQ format still works as plain
        # text continuation, no few-shot example needed since the
        # (A)/(B) structure itself cues the expected answer shape
        return prompt


def already_done(results_path):
    done = set()
    if os.path.exists(results_path):
        with open(results_path) as f:
            for line in f:
                r = json.loads(line)
                done.add((r["item_id"], r["condition"], r["step"]))
    return done


def run_checkpoint(stage_name, checkpoint_info, rows):
    checkpoint_name = checkpoint_info["name"]
    fmt = checkpoint_info["format"]
    results_path = os.path.join(RESULTS_DIR, f"{stage_name}.jsonl")

    done = already_done(results_path)
    total_calls = len(rows) * len(STEPS)
    if len(done) == total_calls:
        print(f"[{stage_name}] already complete ({total_calls} calls) -- skipping.")
        return

    model, tokenizer = load_model(checkpoint_name)

    with open(results_path, "a") as f:
        call_count = 0
        for row in rows:
            for step in STEPS:
                key = (row["item_id"], row["condition"], step)
                if key in done:
                    continue

                mcq_prompt = row["probes"][step]["mcq"]
                letter_map = row["probes"][step]["mcq_letter_map"]

                try:
                    formatted = format_for_scoring(mcq_prompt, fmt, tokenizer)
                    winner_meaning, winner_letter, scores = mcq_choice(
                        model, tokenizer, formatted, letter_map
                    )
                    error = None
                except Exception as e:
                    winner_meaning, winner_letter, scores = None, None, None
                    error = str(e)

                record = {
                    "stage": stage_name,
                    "checkpoint": checkpoint_name,
                    "item_id": row["item_id"],
                    "character": row["character"],
                    "condition": row["condition"],
                    "step": step,
                    "target_property": row["target_property"],
                    "wonky_property": row["wonky_property"],
                    "mcq_prompt": mcq_prompt,
                    "letter_map": letter_map,
                    "winner_meaning": winner_meaning,
                    "winner_letter": winner_letter,
                    "scores": scores,
                    "reliable": step in RELIABLE_STEPS,
                    "error": error,
                }
                f.write(json.dumps(record) + "\n")
                f.flush()

                call_count += 1
                if call_count % 20 == 0:
                    print(f"[{stage_name}] {call_count}/{total_calls - len(done)} new calls done")

    print(f"[{stage_name}] finished -> {results_path}")

    del model
    torch.cuda.empty_cache()


def main():
    with open(PROBES_PATH) as f:
        rows = json.load(f)
    print(f"Loaded {len(rows)} probe rows.")

    if RUN_ALL_CHECKPOINTS:
        for stage_name, info in CHECKPOINTS.items():
            run_checkpoint(stage_name, info, rows)
    else:
        run_checkpoint(CHECKPOINT_TO_RUN, CHECKPOINTS[CHECKPOINT_TO_RUN], rows)


if __name__ == "__main__":
    # Placeholder for mcq_choice function, assuming it's defined elsewhere or will be provided.
    # For now, it will raise a NameError if not defined, but the primary error (load_in_8bit) is addressed.
    def mcq_choice(*args, **kwargs):
        raise NotImplementedError("mcq_choice function is not defined in this snippet.")

    main()

Loaded 96 probe rows.
Loading allenai/OLMo-2-1124-7B ...


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

[pretrained] 20/288 new calls done
[pretrained] 40/288 new calls done
[pretrained] 60/288 new calls done
[pretrained] 80/288 new calls done
[pretrained] 100/288 new calls done
[pretrained] 120/288 new calls done
[pretrained] 140/288 new calls done
[pretrained] 160/288 new calls done
[pretrained] 180/288 new calls done
[pretrained] 200/288 new calls done
[pretrained] 220/288 new calls done
[pretrained] 240/288 new calls done
[pretrained] 260/288 new calls done
[pretrained] 280/288 new calls done
[pretrained] finished -> /content/drive/MyDrive/Colab Notebooks/pragmatics_project/results.json/pretrained.jsonl


In [17]:
"""
Task 3: full inference harness (v2 -- MCQ log-prob scoring).

Runs all 96 rows (24 items x 4 conditions) x 3 diagnostic steps,
for each of the 3 OLMo 2 7B checkpoints, using the MCQ (lettered A/B)
log-prob scoring method validated on item 1's pilot.

script_knowledge and inference_derivation are the two VALIDATED steps
-- both matched predictions cleanly on item 1 pilot.

redundancy_id is still run and saved (so the data exists if you want
it later) but should be treated as UNRELIABLE: repeated fixes (free
text, phrase-level log-prob, letter-randomized MCQ) all still showed
a bias toward "already expected" regardless of content. Flagged in
the output with a `reliable: False` field. Parked as a known
limitation, not used in the main analysis.

Saves incrementally to JSONL so a Colab crash mid-run doesn't lose
everything already computed.
"""

import json
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- 3.1: checkpoint config --------------------------------------------
CHECKPOINTS = {
    "pretrained": {"name": "allenai/OLMo-2-1124-7B", "format": "completion"},
    "sft":        {"name": "allenai/OLMo-2-1124-7B-SFT", "format": "chat"},
    "dpo":        {"name": "allenai/OLMo-2-1124-7B-DPO", "format": "chat"},
}

RUN_ALL_CHECKPOINTS = False
CHECKPOINT_TO_RUN = "dpo"  # used only if RUN_ALL_CHECKPOINTS is False

PROBES_PATH = "/content/drive/MyDrive/Colab Notebooks/pragmatics_project/redundancy_probes.json"
RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

STEPS = ["script_knowledge", "redundancy_id", "inference_derivation"]
RELIABLE_STEPS = {"script_knowledge", "inference_derivation"}  # redundancy_id parked


def load_model(checkpoint_name):
    print(f"Loading {checkpoint_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

    # Use BitsAndBytesConfig for 8-bit loading
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        checkpoint_name,
        quantization_config=quantization_config,
        device_map="auto",
    )
    return model, tokenizer


def format_for_scoring(prompt, fmt, tokenizer):
    """Apply chat-template formatting for chat-format checkpoints;
    pass through unchanged for completion-format (base) checkpoints."""
    if fmt == "chat":
        messages = [{"role": "user", "content": prompt}]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        # base/completion checkpoints: MCQ format still works as plain
        # text continuation, no few-shot example needed since the
        # (A)/(B) structure itself cues the expected answer shape
        return prompt


def already_done(results_path):
    done = set()
    if os.path.exists(results_path):
        with open(results_path) as f:
            for line in f:
                r = json.loads(line)
                done.add((r["item_id"], r["condition"], r["step"]))
    return done


def run_checkpoint(stage_name, checkpoint_info, rows):
    checkpoint_name = checkpoint_info["name"]
    fmt = checkpoint_info["format"]
    results_path = os.path.join(RESULTS_DIR, f"{stage_name}.jsonl")

    done = already_done(results_path)
    total_calls = len(rows) * len(STEPS)
    if len(done) == total_calls:
        print(f"[{stage_name}] already complete ({total_calls} calls) -- skipping.")
        return

    model, tokenizer = load_model(checkpoint_name)

    with open(results_path, "a") as f:
        call_count = 0
        for row in rows:
            for step in STEPS:
                key = (row["item_id"], row["condition"], step)
                if key in done:
                    continue

                mcq_prompt = row["probes"][step]["mcq"]
                letter_map = row["probes"][step]["mcq_letter_map"]

                try:
                    formatted = format_for_scoring(mcq_prompt, fmt, tokenizer)
                    winner_meaning, winner_letter, scores = mcq_choice(
                        model, tokenizer, formatted, letter_map
                    )
                    error = None
                except Exception as e:
                    winner_meaning, winner_letter, scores = None, None, None
                    error = str(e)

                record = {
                    "stage": stage_name,
                    "checkpoint": checkpoint_name,
                    "item_id": row["item_id"],
                    "character": row["character"],
                    "condition": row["condition"],
                    "step": step,
                    "target_property": row["target_property"],
                    "wonky_property": row["wonky_property"],
                    "mcq_prompt": mcq_prompt,
                    "letter_map": letter_map,
                    "winner_meaning": winner_meaning,
                    "winner_letter": winner_letter,
                    "scores": scores,
                    "reliable": step in RELIABLE_STEPS,
                    "error": error,
                }
                f.write(json.dumps(record) + "\n")
                f.flush()

                call_count += 1
                if call_count % 20 == 0:
                    print(f"[{stage_name}] {call_count}/{total_calls - len(done)} new calls done")

    print(f"[{stage_name}] finished -> {results_path}")

    del model
    torch.cuda.empty_cache()


def main():
    # fail loudly BEFORE touching any model/data if mcq_choice isn't
    # actually defined -- prevents a full run silently logging 288
    # null/error rows (this happened once already; don't repeat it)
    try:
        mcq_choice
    except NameError:
        raise RuntimeError(
            "mcq_choice is not defined in this session. Run the "
            "logprob_scoring cell (or upload logprob_scoring.py) "
            "BEFORE running the harness. Aborting before any model load."
        )

    with open(PROBES_PATH) as f:
        rows = json.load(f)
    print(f"Loaded {len(rows)} probe rows.")

    if RUN_ALL_CHECKPOINTS:
        for stage_name, info in CHECKPOINTS.items():
            run_checkpoint(stage_name, info, rows)
    else:
        run_checkpoint(CHECKPOINT_TO_RUN, CHECKPOINTS[CHECKPOINT_TO_RUN], rows)


if __name__ == "__main__":
    main()

Loaded 96 probe rows.
Loading allenai/OLMo-2-1124-7B-DPO ...


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

[dpo] 20/288 new calls done
[dpo] 40/288 new calls done
[dpo] 60/288 new calls done
[dpo] 80/288 new calls done
[dpo] 100/288 new calls done
[dpo] 120/288 new calls done
[dpo] 140/288 new calls done
[dpo] 160/288 new calls done
[dpo] 180/288 new calls done
[dpo] 200/288 new calls done
[dpo] 220/288 new calls done
[dpo] 240/288 new calls done
[dpo] 260/288 new calls done
[dpo] 280/288 new calls done
[dpo] finished -> /content/drive/MyDrive/Colab Notebooks/pragmatics_project/results/dpo.jsonl
